# SI-SDR — ce que la métrique mesure, et son point zéro

Le carnet [04-20](04-20-Separation-Avancee-Qualite.ipynb) mesure trois architectures sur MUSDB18, contre les stems studio : il répond à « quel modèle sépare le mieux », et sa référence est fiable. Mais il ne peut pas répondre à une question plus élémentaire : **que vaut le zéro de la métrique ?** Un SI-SDR de 12 dB est-il bon en soi ? Que mesure un « SI-SDR moyen » quand les tiges n'ont pas la même énergie ?

Ce carnet construit un mélange dont la vérité est **exacte par construction** : deux stems studio que nous additionnons nous-mêmes, avec des gains que nous choisissons. Ça donne un lot de contrôle que MUSDB ne fournit pas — **estimer chaque tige par le mélange lui-même** — et un levier qu'aucune piste réelle ne donne : **déplacer le ratio d'énergie entre tiges et regarder bouger le point zéro**.

## Objectifs d'apprentissage

- Écrire SI-SDR, et savoir ce que son invariance d'échelle change à la lecture d'un chiffre.
- Construire un mélange à vérité exacte à partir de stems studio, avec un ratio d'énergie contrôlé.
- Mesurer le **lot de contrôle** (le mélange comme estimateur) : deux notes opposées sur le même extrait, dont l'écart se creuse avec le déséquilibre des énergies.
- Faire tourner trois moteurs (Demucs, MDX23C, BS-RoFormer) sur ce mélange, et conclure ce que les nombres autorisent — et ce qu'ils n'autorisent pas.

## Prérequis

`torch` + CUDA, `soundfile`, `datasets` (miroir HuggingFace de MUSDB18), `demucs`, `audio-separator` (qui porte MDX23C et BS-RoFormer). Les poids se téléchargent au premier appel (~640 Mo pour BS-RoFormer).

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False                 # True pour mode batch MCP
BATCH_MODE = "false"

# Corpus de reference (miroir HuggingFace, comme 04-20)
dataset_id = "danjacobellis/musdb18HQ"
split = "train"
clip_seconds = 20                        # extrait utilise

# Melange a verite exacte : deux stems studio, deux gains choisis
gain_vocals = 0.62
gain_accomp = 0.48

# Balayage du ratio d'energie (section 4)
ratios_db = [-12, -6, -3, 0, 3, 6, 12]

# Architectures (registre natif demucs + registre audio-separator)
demucs_model = "htdemucs"
mdx23c_ckpt = "MDX23C-8KFFT-InstVoc_HQ.ckpt"
roformer_ckpt = "model_bs_roformer_ep_317_sdr_12.9755.ckpt"

# Device
device = "cuda"                      # "cuda" ou "cpu"

In [2]:
# Parameters
notebook_mode = "batch"
skip_widgets = True
BATCH_MODE = "true"

Les paramètres fixent l'extrait MUSDB18 (20 s), les **deux gains** qui fabriquent le mélange — `0,62` sur les voix, `0,48` sur l'accompagnement — et le balayage du ratio d'énergie de la section 4. Les trois moteurs sont nommés par leur checkpoint exact : `htdemucs` (registre natif demucs), `MDX23C-8KFFT-InstVoc_HQ` et `model_bs_roformer_ep_317_sdr_12.9755` (registre `audio-separator`).

## 1. La métrique, et son invariance d'échelle

SI-SDR (scale-invariant signal-to-distortion ratio) projette d'abord l'estimation sur la référence avec le gain optimal α, puis mesure le résidu :

$$\mathrm{SI\text{-}SDR}(\hat{s}, s) = 10 \log_{10} \frac{\|\alpha s\|^2}{\|\hat{s} - \alpha s\|^2}, \qquad \alpha = \frac{\hat{s}^\top s}{\|s\|^2}$$

La conséquence est immédiate et souvent oubliée en lisant un tableau : **multiplier l'estimation par n'importe quelle constante ne change pas le score**. La métrique ne voit pas une erreur de niveau — elle voit une erreur de *direction*. Une cellule qui se lit vite mais qui se vérifie :

In [3]:
import numpy as np


def si_sdr(est: np.ndarray, ref: np.ndarray) -> float:
    """SI-SDR en dB, apres projection au gain optimal.

    est, ref : tableaux 1D ou 2D (canaux en colonnes) ; aparies en longueur.
    """
    est = est.reshape(-1).astype(np.float64)
    ref = ref.reshape(-1).astype(np.float64)
    n = min(len(est), len(ref))
    est, ref = est[:n], ref[:n]
    est = est - est.mean()
    ref = ref - ref.mean()
    alpha = float(np.dot(est, ref) / (np.dot(ref, ref) + 1e-12))
    target = alpha * ref
    noise = est - target
    return float(10 * np.log10((np.dot(target, target) + 1e-12)
                               / (np.dot(noise, noise) + 1e-12)))


# Verifications sur un signal connu, avant toute mesure de modele
rng = np.random.default_rng(42)
ref = rng.standard_normal(44100).astype(np.float64)          # 1 s de bruit blanc
bruit = 0.1 * rng.standard_normal(44100).astype(np.float64)
est_scale = 3.7 * ref                                        # bonne direction, gain faux
est_noisy = ref + bruit                                      # bonne echelle, direction bruitee

print(f"SI-SDR(est = 3,7 x reference, ref)      = {si_sdr(est_scale, ref):>8.2f} dB")
print(f"SI-SDR(est = 0,001 x reference, ref)   = {si_sdr(0.001 * ref, ref):>8.2f} dB")
print(f"SI-SDR(est = reference + bruit, ref)   = {si_sdr(est_noisy, ref):>8.2f} dB")
print(f"SI-SDR(reference, reference)           = {si_sdr(ref, ref):>8.2f} dB (residu nul)")

SI-SDR(est = 3,7 x reference, ref)      =   177.84 dB
SI-SDR(est = 0,001 x reference, ref)   =   106.48 dB
SI-SDR(est = reference + bruit, ref)   =    20.00 dB
SI-SDR(reference, reference)           =   166.48 dB (residu nul)


### Lecture du résultat

Multiplier l'estimation par 3,7 — ou par 0,001 — la laisse **colinéaire** à la référence : le résidu tombe au plancher de la précision flottante, et les deux lignes affichent des scores au-delà de 100 dB. Ces valeurs ne sont pas des performances comparables entre elles : elles signifient toutes « aucune erreur de direction », au plafond numérique de la formule — la ligne `reference, reference` en fixe le niveau. Ajouter un bruit de 10 % d'amplitude, en revanche, coûte **exactement 20 dB** — dix fois moins d'énergie de bruit que de signal, lue directement par la métrique. Aveugle au niveau, sensible à la direction : c'est le contrat de lecture de tout le carnet.

## 2. Un mélange à vérité exacte, et un gain qu'on choisit

Une piste MUSDB18 nous donne ses stems studio séparés. Nous ne les utilisons **pas** comme une vérité mesurée après coup : nous **fabriquons** le mélange nous-mêmes, `0,62 × voix + 0,48 × accompagnement`. La vérité de chaque tige est donc exacte par construction — le seul écart est l'arrondi PCM 16 bits à l'écriture, mesuré plus bas.

L'accompagnement est la somme `drums + bass + other`, comme dans 04-20 — ce qui neutralise la différence de granularité entre moteurs à 4 stems et moteurs à 2 stems.

In [4]:
import io
import os
import time
import logging
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
for _nom in ("separator", "audio_separator", "mdxc_separator", "common_separator"):
    logging.getLogger(_nom).setLevel(logging.WARNING)

import numpy as np
import soundfile as sf
import torch
from datasets import load_dataset

# Racine de serie : depuis le decoupage en sous-series (#23553), le cwd de ce
# carnet est son sous-dossier (04-Musique/), alors que les sorties vivent a la
# racine de 04-Applications. On remonte jusqu'a ce repertoire nomme.
SERIES_ROOT = Path.cwd()
while SERIES_ROOT.name != '04-Applications' and len(SERIES_ROOT.parts) > 1:
    SERIES_ROOT = SERIES_ROOT.parent
OUTPUT_DIR = SERIES_ROOT / "outputs" / "04-21"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch {torch.__version__} - CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_properties(0).name}")
print(f"Debut du carnet : {datetime.now().isoformat(timespec='seconds')}")


def decoder_audio(audio, max_seconds=None):
    """Decode une colonne audio HF en (echantillons, frequence)."""
    data, sr = sf.read(io.BytesIO(audio["bytes"]))
    if max_seconds:
        data = data[: int(max_seconds * sr)]
    return data.astype(np.float32), sr


ds = load_dataset(dataset_id, split=split, streaming=True)
par_piste = {}
for row in ds:
    nom = row["path"].rsplit("/", 2)[1]
    par_piste.setdefault(nom, {})[row["instrument"]] = decoder_audio(row["audio"], clip_seconds)
    if len(par_piste[nom]) == 5:
        break
nom_piste = nom
piste = par_piste[nom_piste]
sr = piste["mixture"][1]

voix = piste["vocals"][0]                                  # (n, 2)
accomp = sum(piste[k][0] for k in ("drums", "bass", "other"))
melange = gain_vocals * voix + gain_accomp * accomp

# Ecriture PCM 16 : c'est le format que les moteurs lisent. La relecture
# mesure l'ecart exact entre la verite construite et la verite ecrite.
sf.write(OUTPUT_DIR / "melange.wav", melange, sr, subtype="PCM_16")
melange_pcm, _ = sf.read(OUTPUT_DIR / "melange.wav")

def rms_db(x):
    return 10 * np.log10(float(np.mean(x ** 2)) + 1e-12)

print(f"Piste : {nom_piste} - extrait {clip_seconds} s a {sr / 1000:.1f} kHz")
print(f"RMS voix (x{gain_vocals})            : {rms_db(gain_vocals * voix):>7.2f} dB")
print(f"RMS accompagnement (x{gain_accomp})  : {rms_db(gain_accomp * accomp):>7.2f} dB")
print(f"ecart d'energie entre les deux tiges : {rms_db(gain_vocals * voix) - rms_db(gain_accomp * accomp):>7.2f} dB")
print(f"Arrondi PCM (verite ecrite vs construite) : "
      f"{rms_db(melange_pcm) - rms_db(melange):+.4f} dB")

PyTorch 2.14.0+cu126 - CUDA : True
GPU : NVIDIA GeForce RTX 3090
Debut du carnet : 2026-11-16T09:35:22


Piste : Alexander Ross - Velvet Curtain - extrait 20 s a 44.1 kHz
RMS voix (x0.62)            :  -41.71 dB
RMS accompagnement (x0.48)  :  -42.85 dB
ecart d'energie entre les deux tiges :    1.14 dB
Arrondi PCM (verite ecrite vs construite) : +0.0000 dB


### Lecture du résultat

Le mélange est fabriqué, pas trouvé : ses deux contributions sont connues au flottant près, et l'arrondi PCM 16 bits de l'écriture — le seul écart introduit entre la vérité construite et la vérité sur disque — se mesure en millièmes de dB, donc négligeable pour la suite. La ligne la plus importante est **l'écart d'énergie entre les deux tiges** : 1,14 dB après les gains choisis, en faveur de la voix. C'est ce seul chiffre qui fixera le zéro de la section suivante — et le balayage de la section 4 montrera ce qui arrive quand on le déplace.

## 3. Le lot de contrôle : le mélange comme estimateur

Avant de mesurer un modèle, mesurons le **pire estimateur honnête** : ne rien séparer du tout, rendre le mélange brut comme estimation de chaque tige. C'est le zéro *de cet extrait* — le score qu'obtiendrait un moteur qui ne fait rien. Tout score de modèle se lit comme un écart à **ce** zéro-là, pas comme une hauteur absolue.

In [5]:
# References exactes : la contribution de chaque tige au melange construit
ref_voix = gain_vocals * voix
ref_accomp = gain_accomp * accomp

# Lot de controle : le melange lui-meme, tel quel, comme estimation
zero_voix = si_sdr(melange_pcm, ref_voix)
zero_accomp = si_sdr(melange_pcm, ref_accomp)

print("Lot de controle - le melange rendu sans aucune separation :")
print(f"  SI-SDR(melange, voix)             = {zero_voix:>8.2f} dB")
print(f"  SI-SDR(melange, accompagnement)   = {zero_accomp:>8.2f} dB")
print(f"\n  moyenne naive des deux            = {(zero_voix + zero_accomp) / 2:>8.2f} dB")
print(f"  meme signal, meme estimateur, deux notes : "
      f"ecart de {zero_accomp - zero_voix:.1f} dB")

Lot de controle - le melange rendu sans aucune separation :
  SI-SDR(melange, voix)             =     1.14 dB
  SI-SDR(melange, accompagnement)   =    -1.13 dB

  moyenne naive des deux            =     0.00 dB
  meme signal, meme estimateur, deux notes : ecart de -2.3 dB


### Lecture du résultat

C'est la cellule centrale du carnet. **Le même estimateur — le mélange, non modifié — reçoit deux notes opposées** : +1,14 dB sur la voix (la tige la plus énergique de cet extrait), -1,13 dB sur l'accompagnement. Ici l'extrait est presque équilibré (1,14 dB d'écart), donc les deux notes sont quasi symétriques — et leur moyenne naive vaut exactement 0,00 dB : un chiffre qui ne correspond à aucune écoute, puisqu'il ne dit ni que les voix sont isolées, ni qu'elles sont perdues. La section 4 montre la même mesure quand le déséquilibre croît : le zéro de la voix parcourt à lui seul ~24 dB. Le score ne mesure donc pas seulement la qualité de l'estimateur — il mesure **où se situe l'énergie de la tige demandée par rapport à l'autre**, et une « moyenne SI-SDR » entre tiges compense des notes qui ne racontent pas la même histoire.

## 4. Le point zéro se déplace avec le ratio d'énergie

Le gain de l'autre tige est un curseur : atténuer l'accompagnement de 12 dB ne rend pas le mélange « meilleur » — il est ce qu'il est — mais cela **déplace le zéro de la métrique** pour la voix. Aucune piste réelle ne permet ce geste ; un mélange fabriqué, si.

In [6]:
print("Attenuation de l'accompagnement | zero de la metrique pour la voix")
print("-" * 66)
for r_db in ratios_db:
    r = 10 ** (r_db / 20)                      # ratio d'amplitude
    mix_r = gain_vocals * voix + gain_accomp * r * accomp
    z = si_sdr(mix_r, gain_vocals * voix)
    print(f"{r_db:>+12d} dB                   | {z:>8.2f} dB")

Attenuation de l'accompagnement | zero de la metrique pour la voix
------------------------------------------------------------------


         -12 dB                   |    13.14 dB
          -6 dB                   |     7.14 dB
          -3 dB                   |     4.14 dB


          +0 dB                   |     1.14 dB


          +3 dB                   |    -1.86 dB
          +6 dB                   |    -4.85 dB
         +12 dB                   |   -10.85 dB


### Lecture du résultat

Chaque ligne du balayage est le score du même « ne rien faire » face aux mêmes voix — **seul le curseur change**. À -12 dB d'accompagnement, rendre le mélange est déjà un estimateur passable des voix ; à +12 dB, c'est une catastrophe. Entre les deux, le zéro parcourt ~24 dB : exactement l'ordre de grandeur qui sépare un mauvais moteur d'un bon sur un tableau de benchmark. Autrement dit, **une partie de l'écart qu'on lit entre deux modèles, sur deux extraits différents, peut venir du ratio d'énergie des extraits, pas des modèles**. C'est pourquoi 04-20 compare ses moteurs sur la même piste, et pourquoi les valeurs absolues d'un tableau ne se comparent pas entre pistes.

## 5. Trois moteurs sur le même mélange

Il reste à placer de vrais moteurs sur cette échelle. `htdemucs` (4 stems, registre natif demucs) rend `vocals` directement ; MDX23C et BS-RoFormer (registre `audio-separator`, 2 stems) rendent `vocals` + `instrumental`. L'accompagnement de Demucs est la somme `drums + bass + other`, celle des deux autres est leur `instrumental` — la même convention qu'en 04-20 pour neutraliser la granularité.

In [7]:
from demucs.pretrained import get_model
from demucs.apply import apply_model
from audio_separator.separator import Separator

resultats = {}

# --- Demucs htdemucs (registre natif, 4 stems) ---
t0 = time.time()
m = get_model(demucs_model).to(device).eval()
with torch.no_grad():
    x = torch.from_numpy(melange_pcm.T).float().unsqueeze(0).to(device)
    out = apply_model(m, x, device=device, progress=False)[0].cpu().numpy()  # (4, n, ch)
dem = {name: out[i].T for i, name in enumerate(m.sources)}
del m
torch.cuda.empty_cache()
resultats["htdemucs"] = {
    "voix": si_sdr(dem["vocals"], ref_voix),
    "accompagnement": si_sdr(dem["drums"] + dem["bass"] + dem["other"], ref_accomp),
    "secondes": time.time() - t0,
}

# --- MDX23C et BS-RoFormer (registre audio-separator, 2 stems) ---
MODELS_DIR = OUTPUT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
for label, ckpt in (("MDX23C-8KFFT", mdx23c_ckpt),
                    ("BS-RoFormer-1297", roformer_ckpt)):
    t0 = time.time()
    sep = Separator(log_level=logging.WARNING,
                    model_file_dir=str(MODELS_DIR),
                    output_dir=str(OUTPUT_DIR / label),
                    output_format="WAV")
    sep.load_model(model_filename=ckpt)
    fichiers = sep.separate(str(OUTPUT_DIR / "melange.wav"))
    stems = {}
    for f in fichiers:
        p = Path(f) if Path(f).is_absolute() else OUTPUT_DIR / label / f
        nom = p.name.lower()
        data, sr_out = sf.read(str(p))
        if "vocal" in nom:
            stems["vocals"] = data
        else:
            stems["instrumental"] = data
    resultats[label] = {
        "voix": si_sdr(stems["vocals"], ref_voix),
        "accompagnement": si_sdr(stems["instrumental"], ref_accomp),
        "secondes": time.time() - t0,
    }

# --- Tableau : chaque moteur face a son zero (unites en dB) ---
print(f"{'moteur':<18}{'voix':>9}{'zéro':>9}{'écart':>9}"
      f"{'accomp.':>9}{'zéro':>9}{'écart':>9}{'durée':>8}")
print("-" * 80)
for label, r in resultats.items():
    print(f"{label:<18}"
          f"{r['voix']:>9.2f}{zero_voix:>9.2f}{r['voix'] - zero_voix:>9.2f}"
          f"{r['accompagnement']:>9.2f}{zero_accomp:>9.2f}"
          f"{r['accompagnement'] - zero_accomp:>9.2f}"
          f"{r['secondes']:>7.1f}s")

  0%|          | 0.00/3.54k [00:00<?, ?iB/s]

28.3kiB [00:00, 18.8MiB/s]                  

  0%|          | 0.00/448M [00:00<?, ?iB/s]

  0%|          | 999k/448M [00:00<00:45, 9.85MiB/s]

  0%|          | 2.08M/448M [00:00<00:43, 10.3MiB/s]

  1%|          | 3.12M/448M [00:00<00:48, 9.17MiB/s]

... (1,944 outputs left out)

 86%|████████▌ | 6/7 [00:04<00:00,  1.29it/s]

100%|██████████| 7/7 [00:05<00:00,  1.29it/s]

100%|██████████| 7/7 [00:05<00:00,  1.28it/s]

moteur                 voix     zéro    écart  accomp.     zéro    écart   durée
--------------------------------------------------------------------------------
htdemucs              21.40     1.14    20.26    20.23    -1.13    21.36    2.6s
MDX23C-8KFFT          24.54     1.14    23.40    23.38    -1.13    24.51   83.5s
BS-RoFormer-1297      27.29     1.14    26.15    26.12    -1.13    27.25  167.6s


### Lecture du résultat

La colonne à lire n'est pas la note absolue — c'est **l'écart au zéro**, ce que le moteur ajoute au « ne rien faire ». Trois constats mesurés. D'abord, l'écart au zéro dépasse **20 dB pour les trois moteurs sur les deux tiges** : sur ce mélange à deux sources, séparer rapporte un ordre de grandeur entier, quelle que soit la tige. Ensuite, les durées : 2,6 s pour `htdemucs` contre 83,5 s et 167,6 s pour MDX23C et BS-RoFormer — ~6 dB de voix en plus, payés au prix de plus de soixante fois le temps de calcul. Enfin, le classement (BS-RoFormer devant MDX23C devant Demucs) est **le même sur les deux tiges** et **le même que sur les leaderboards MUSDB publics** — mais la section 4 a montré que le zéro se déplace de 24 dB avec le ratio d'énergie : sur un extrait réellement déséquilibré, une inversion de colonne reste possible, et c'est l'objet de l'exercice 3.

## 6. Ce que ces nombres autorisent à dire

Un dernier geste de rigueur : opposer à ce qui précède le classement **publié** sur MUSDB18-HQ (le corpus d'entraînement de ces moteurs), pour vérifier ce qui se transpose et ce qui ne se transpose pas.

In [8]:
# SDR vocals publie sur MUSDB18-HQ (leaderboards MDX23 / audio-separator) :
# BS-RoFormer-1297 ~ 12,98 dB (c'est dans le nom du checkpoint), MDX23C ~ 12 dB,
# htdemucs ~ 8,1 dB. Notre extrait n'est PAS MUSDB egal a lui-meme : il est
# re-pondere par deux gains choisis. On compare l'ORDRE, pas les valeurs.
publie = {"BS-RoFormer-1297": 12.98, "MDX23C-8KFFT": 12.0, "htdemucs": 8.1}

ordre_ici = sorted(resultats, key=lambda k: -resultats[k]["voix"])
ordre_publie = sorted(publie, key=lambda k: -publie[k])

print("ordre sur notre melange (voix) :", " > ".join(ordre_ici))
print("ordre publie MUSDB18-HQ (voix) :", " > ".join(ordre_publie))
print()
print("l'ordre se transpose :", ordre_ici == ordre_publie)
for k in ordre_ici:
    print(f"  {k:<18} ici {resultats[k]['voix']:>7.2f} dB   | publie {publie[k]:>6.2f} dB")

ordre sur notre melange (voix) : BS-RoFormer-1297 > MDX23C-8KFFT > htdemucs
ordre publie MUSDB18-HQ (voix) : BS-RoFormer-1297 > MDX23C-8KFFT > htdemucs

l'ordre se transpose : True
  BS-RoFormer-1297   ici   27.29 dB   | publie  12.98 dB
  MDX23C-8KFFT       ici   24.54 dB   | publie  12.00 dB
  htdemucs           ici   21.40 dB   | publie   8.10 dB


### Lecture du résultat

L'ordre tient ; les valeurs absolues, non — et ce n'est pas un accident de mesure, c'est la structure même de l'expérience : notre mélange re-pondéré n'est pas le mélange MUSDB sur lequel les chiffres publiés sont calculés, et un extrait de 20 secondes ne se compare pas à une moyenne sur l'ensemble de test complet du corpus. C'est la deuxième leçon du carnet : **un tableau de SI-SDR est un instrument de comparaison relative, pas un thermomètre universel**. Le classement entre moteurs évalués dans les mêmes conditions est robuste ; transposer la valeur absolue d'un corpus à l'autre, ou d'une tige à l'autre, ne l'est pas.

## Exercices

Les trois exercices suivent la progression du carnet : le gain optimal (section 1), le point zéro (sections 3-4), le classement (section 5). Chaque cellule s'exécute telle quelle — elle affiche seulement l'énoncé ; c'est **votre** code qui remplace la ligne `TODO`.

### Exercice 1 — le gain oublié

SI-SDR ne voit pas le niveau. Écrivez `gain_optimal(est, ref)` qui retourne le α de la formule, puis vérifiez sur deux stems du carnet que corriger le gain d'une estimation **ne change pas** son SI-SDR — et dites en une ligne quel défaut un ingénieur du son doit donc attraper avec ses oreilles, pas avec cette métrique.

**Etape 1** : projeter `est` sur `ref` (produit scalaire sur énergie de la référence). **Etape 2** : appliquer α à `ref`, mesurer le résidu. **Etape 3** : comparer `si_sdr(est, ref)` et `si_sdr(alpha * est, ref)` et conclure.

In [9]:
def gain_optimal(est: np.ndarray, ref: np.ndarray) -> float:
    """Alpha de la formule SI-SDR : le gain qui projette est sur ref.

    est, ref : memes formes que si_sdr().
    """
    # TODO etudiant : produit scalaire de est sur ref, divise par l'energie de ref
    return None  # TODO etudiant


print("Exercice 1 a completer :")
print(" - calculer alpha sur (melange_pcm, ref_voix)")
print(" - verifier que si_sdr(alpha * melange_pcm, ref_voix) == si_sdr(melange_pcm, ref_voix)")
print(" - ecrire la conclusion sur ce que la metrique ne voit pas")

Exercice 1 a completer :
 - calculer alpha sur (melange_pcm, ref_voix)
 - verifier que si_sdr(alpha * melange_pcm, ref_voix) == si_sdr(melange_pcm, ref_voix)
 - ecrire la conclusion sur ce que la metrique ne voit pas


### Exercice 2 — déplacer le point zéro

Le zéro de la voix dépend de l'énergie de l'autre tige. Écrivez `zero_voix_pour(attenuation_db)` qui fabrique le mélange où l'accompagnement est atténué de `attenuation_db` et retourne `si_sdr(mélange, ref_voix)` — puis trouvez, par balayage fin (pas de 0,5 dB), l'atténuation pour laquelle le zéro vaut exactement **0 dB**. C'est le point où « ne rien faire » devient un estimateur ni bon ni mauvais.

**Etape 1** : convertir dB en ratio d'amplitude. **Etape 2** : construire le mélange atténué. **Etape 3** : balayer autour de la valeur trouvée en section 4 pour resserrer à ±0,5 dB.

In [10]:
def zero_voix_pour(attenuation_db: float) -> float:
    """SI-SDR du melange (non separe) face aux voix, accompagnement attenue.

    attenuation_db : attenuation appliquee a gain_accomp * accomp (negatif = plus bas).
    """
    # TODO etudiant : ratio d'amplitude, melange, si_sdr face a ref_voix
    return None  # TODO etudiant


print("Exercice 2 a completer :")
print(" - balayer attenuation_db de -12 a +12 par pas de 0,5")
print(" - trouver l'attenuation ou zero_voix_pour(...) coupe 0 dB")
print(" - comparer au tableau de la section 4")

Exercice 2 a completer :
 - balayer attenuation_db de -12 a +12 par pas de 0,5
 - trouver l'attenuation ou zero_voix_pour(...) coupe 0 dB
 - comparer au tableau de la section 4


### Exercice 3 — le classement tient-il tige par tige ?

La section 5 a comparé les moteurs sur la tige voix. Écrivez `classement(resultats, tige)` qui trie les moteurs par SI-SDR décroissant sur **une tige donnée** (`"voix"` ou `"accompagnement"`), puis répondez : le classement des trois moteurs est-il le même sur les deux tiges ? Si non, lequel change de place, et pourquoi la réponse à cette question interdit-elle de publier un « classement moyen » ?

In [11]:
def classement(resultats: dict, tige: str) -> list[str]:
    """Liste des moteurs, du meilleur au moins bon sur la tige donnee.

    resultats : dictionnaire {moteur: {tige: si_sdr, ...}} (celui de la section 5).
    tige : "voix" ou "accompagnement".
    """
    # TODO etudiant : tri decroissant sur resultats[moteur][tige]
    return None  # TODO etudiant


print("Exercice 3 a completer :")
print(" - classement(resultats, 'voix') et classement(resultats, 'accompagnement')")
print(" - identifier la ou les moteurs qui changent de place")
print(" - conclure sur la legitimite d'un classement moyen")

Exercice 3 a completer :
 - classement(resultats, 'voix') et classement(resultats, 'accompagnement')
 - identifier la ou les moteurs qui changent de place
 - conclure sur la legitimite d'un classement moyen


## Conclusion

Ce carnet n'a pas comparé des moteurs — il a étalonné l'instrument qui les compare.

- **SI-SDR est invariant d'échelle** : un gain global, même ridicule, ne coûte rien ; une direction bruitée coûte tout. La métrique ne juge pas le niveau.
- **Son point zéro n'est pas zéro** : le mélange non séparé marque +1,1 dB sur la tige forte et -1,1 dB sur la tige faible de notre extrait, et l'écart se creuse avec le déséquilibre des énergies (±12 dB aux bornes du balayage). La note absolue dit autant l'énergie de la tige que la qualité de l'estimateur.
- **Le point zéro se déplace avec le ratio d'énergie** : ~24 dB parcourus sur le balayage, l'ordre de grandeur d'un écart entre moteurs. Comparer des SI-SDR mesurés sur des extraits différents, c'est comparer des échelles différentes.
- **Ce qui se transpose, c'est l'ordre** : sur ce mélange re-pondéré, les trois moteurs se classent comme sur MUSDB18-HQ. Les valeurs absolues ne se transplantent pas ; le classement à conditions égales, si.

La règle de lecture qui en découle : toujours mesurer le **lot de contrôle** (le mélange comme estimateur), toujours rapporter **l'écart au zéro** plutôt que la note absolue, et toujours comparer **tige par tige**.

## Pour aller plus loin

- [04-20 — Séparation avancée, qualité mesurée sur MUSDB18](04-20-Separation-Avancee-Qualite.ipynb) : la comparaison des moteurs sur le corpus de référence.
- [04-19 — Évaluation MOS](../04-Voix/04-19-Evaluation-MOS.ipynb) : ce que l'écoute ajoute à la métrique.
- [02-4 — Demucs, le geste de base](../../02-Advanced/02-4-Demucs-Source-Separation.ipynb) : séparer, écouter, remixer.